# Comparative reporting

Step-by-step notebook for comparing flat evaluation runs with `soda_mmqc.reporting`.

We build one comparison view at a time. We load **both models** up front; later steps will compare models as well as prompts.

Views so far: **Structure (Layer S)**, **Applicability (Layer 1)** comparisons, and a **culprit table** for one `(model, prompt)` pair.

| | |
|--|--|
| **Check** | `micrograph-scale-bar` |
| **Models** | `gpt-5-mini-2025-08-07`, `gpt-5` |
| **Prompts** | `prompt.1`, `prompt.2`, `prompt.3` |

Run with the project venv (`uv sync`) from the repo root.

## Setup

In [ ]:
from __future__ import annotations

import pandas as pd

from soda_mmqc.reporting import (
    load_flat_runs,
    plot_comparison_layer1,
    plot_comparison_layer_s,
    show_layer1_errors,
    summarize_runs,
)
from soda_mmqc.reporting.styles import LAYER1_TITLE, LAYER_S_TITLE

CHECKLIST = "fig-checklist"
CHECK = "micrograph-scale-bar"
MODELS = ["gpt-5-mini-2025-08-07", "gpt-5"]
PROMPTS = ["prompt.1", "prompt.2", "prompt.3"]


LAYER_S_FIG_HEIGHT = 500
LAYER_S_FIG_WIDTH = 500
LAYER1_FIG_HEIGHT = 900
LAYER1_FIG_WIDTH = 1200

2026-06-12 17:51:07 - INFO - ✅ OpenAI API provider configured


### Load runs and summarize

Load all `(model, prompt)` pairs we will need for prompt and model contrasts. One `RunSummary` per pair. Layer S counts live in `by_list_row_counts` (keyed by predictive list, usually `outputs`).

In [2]:
runs = load_flat_runs(
    CHECKLIST,
    CHECK,
    models=MODELS,
    prompts=PROMPTS,
)
summaries = summarize_runs(runs)

pd.DataFrame(
    [
        {
            "model": run.model,
            "prompt": run.prompt,
            "records": len(run.records),
            "by_list_keys": ", ".join(
                summaries[run.model, run.prompt].by_list_keys
            ),
        }
        for run in runs
    ]
)

,model,prompt,records,by_list_keys
0,gpt-5-mini-2025-08-07,prompt.1,14,outputs
1,gpt-5-mini-2025-08-07,prompt.2,14,outputs
2,gpt-5-mini-2025-08-07,prompt.3,14,outputs
3,gpt-5,prompt.1,14,outputs
4,gpt-5,prompt.2,14,outputs
5,gpt-5,prompt.3,14,outputs


## Structural reporting

Reporting whether the lists of objects, ie. the panels, are correct, missing or spurious on this check.

In [7]:
# Selectors for the current comparison steps
MODEL = "gpt-5"

In [8]:
fig_layer_s = plot_comparison_layer_s(
    summaries,
    compare="prompt",
    model=MODEL,
)
if fig_layer_s is None:
    print(f"No {LAYER_S_TITLE} data for model={MODEL}")
else:
    fig_layer_s.update_layout(
        height=LAYER_S_FIG_HEIGHT,
        width=LAYER_S_FIG_WIDTH,
        autosize=True,
        margin=dict(l=40, r=20, t=50, b=40),
    )
    fig_layer_s.show()

## Applicability reporting(Layer 1)



### Comparing prompts

In [11]:
MODEL = "gpt-5"

In [12]:
fig_layer1_prompt = plot_comparison_layer1(
    summaries,
    compare="prompt",
    model=MODEL,
)
fig_layer1_prompt.update_layout(
    height=LAYER1_FIG_HEIGHT,
    width=LAYER1_FIG_WIDTH,
    autosize=True,
    margin=dict(l=40, r=20, t=50, b=120),
)
fig_layer1_prompt.show()

### Comparing models


In [15]:
# Selectors for the current comparison steps
PROMPT = "prompt.2"  # scale-bar applicability outliers on this check

In [16]:
fig_layer1_model = plot_comparison_layer1(
    summaries,
    compare="model",
    prompt=PROMPT,
)
fig_layer1_model.update_layout(
    height=LAYER1_FIG_HEIGHT,
    width=LAYER1_FIG_WIDTH,
    autosize=True,
    margin=dict(l=40, r=20, t=50, b=120),
)
fig_layer1_model.show()

### Applicability culprits

In [18]:
# Selectors for the current comparison steps
MODEL = "gpt-5"
PROMPT = "prompt.2"  # scale-bar applicability outliers on this check
CULPRIT_LAYER1 = "spurious_applicable"  # initial table filter; change in footer


In [19]:
summary_selected = summaries[MODEL, PROMPT]

show_layer1_errors(
    summary_selected,
    layer1=CULPRIT_LAYER1,
    caption=(
        f"Layer 1 culprits — {CHECK} / {MODEL} / {PROMPT} "
        f"(initial filter: {CULPRIT_LAYER1})"
    ),
)

Loading ITables v2.8.1 from the init_notebook_mode cell... (need help?)
